# ***Data Cleaning + Regressions***

In [1]:
# Import packages
import numpy as np
import pandas as pd
import os 
from scipy.stats import chi2_contingency
import scipy.stats as stats
import statsmodels.formula.api as smf

In [2]:
# Set directory for raw data, download raw data
brfss_raw_dir = '{}/data/BRFSS2023.csv'.format(os.getcwd())
brfss_raw_df = pd.read_csv(brfss_raw_dir)

In [3]:
# Only select columns of interest
brfss_df = brfss_raw_df[['addepev3', 'firearm5', '_ment14d', '_age80', 'sexvar', '_racegr3', 'primins1', 'educa', 'income3', 'loadulk2', 'children', '_state']]

# Create a new data table from the subsetted data
brfss_dir = '{}/data/BRFSS_subset.csv'.format(os.getcwd())
brfss_df.to_csv(path_or_buf = brfss_dir, index=False)

In [4]:
# List columns and examine first few rows
print(brfss_df.columns.tolist())
brfss_df.head()

['addepev3', 'firearm5', '_ment14d', '_age80', 'sexvar', '_racegr3', 'primins1', 'educa', 'income3', 'loadulk2', 'children', '_state']


,addepev3,firearm5,_ment14d,_age80,sexvar,_racegr3,primins1,educa,income3,loadulk2,children,_state
0,2.0,NaN,1,80,2,1.0,3.0,5.0,99.0,NaN,88.0,1
1,1.0,NaN,1,80,2,1.0,3.0,5.0,99.0,NaN,88.0,1
2,2.0,NaN,2,80,2,2.0,3.0,4.0,2.0,NaN,88.0,1
3,1.0,NaN,1,78,2,1.0,3.0,5.0,99.0,NaN,88.0,1
4,1.0,NaN,1,76,2,1.0,3.0,5.0,7.0,NaN,88.0,1


# **Clean Variables - addepev3, firearm5, loadulk2, _ment14d, _racegr3, primins1, educa, income3, children** 

In [5]:
# ADDEPEV3 (Depression): Set No (2) to No (0)
                       # Set Don't know (7) and Refused (9) to missing
brfss_df['addepev3'] = brfss_df['addepev3'].replace({2:0,
                                                     7: np.nan, 9: np.nan})

# FIREARM5 (Firearms): Set No (2) to No (0)
                     # Don't Know (7) and Refused (9) to missing
brfss_df["firearm5"] = brfss_df["firearm5"].replace({2:0,
                                                     7: np.nan, 9: np.nan})

# LOADULK2 (Unlocked loaded firearms): Set No (2) to No (0) 
                                     # Don't know (7) and Refused (9) to missing 
brfss_df["loadulk2"] = brfss_df["loadulk2"].replace({2:0,
                                                     7: np.nan, 9: np.nan})

# _AGE80 (Age collapsed above 80): No Missingness

# SEXVAR (Sex): No Missingness

# _MENT14D (Mental health status): Set Don't know/Not Sure/Refused (9) to missing 
brfss_df["_ment14d"] = brfss_df["_ment14d"].replace({9: np.nan})

# _RACEGR3 (Racial group): Set Don't know/Not Sure/Refused (9) to missing
brfss_df["_racegr3"] = brfss_df["_racegr3"].replace({9: np.nan})

# PRIMINS1 (Insurance group): Set employer-sponsored (1) and other private (2) to private (1).
                            # Set Medicare (3), Medigap (4), Medicaid (5), and CHIP (6) to public (2). 
                            # Set military (7), Indian Health Services (8), state-sponsored (9), and other government (10) to other (3)
                            # Set no coverage (88) to uninsured (4) 
                            # Set Don't know/Not Sure (77) and Refused (99) to missing
brfss_df["primins1"] = brfss_df["primins1"].replace({2:1,
                                                     3:2, 4:2, 5:2, 6:2,
                                                     7:3, 8:3, 9:3, 10:3,
                                                     88:4,
                                                     77:np.nan, 99:np.nan})

# EDUCA (Education level): Set never attended (1), elementary (2), some high school (3), and high school/GED (4) to high school or less (1)
                         # Set some college/trade school (5) to some college (2) 
                         # Set 4-year degreee or more (6) to Bachelor's or more (3) 
                         # Set Refused (9) to missing  
brfss_df["educa"] = brfss_df["educa"].replace({2:1, 3:1, 4:1,
                                               5:2,
                                               6:3,
                                               9:np.nan})

# INCOME3 (Income categories): Set less than 10k (1), less than 15k (2), and less than 20k (3) to less than 20k (1)
                             # Set less than 25k (4), less than 35k (5), and less than 50k (6) to 20k - 50k (2) 
                             # Set less than 75k (7) and less than 100k (8) to 50k - 100k (3) 
                             # Set less than 150k (9) and less than 200k (10) to 100k - 200k (4)
                             # Set 200k or more (11) to 200k or more (5) 
                             # Set Don't know/Not sure (77) and Refused (99) to missing  
brfss_df["income3"] = brfss_df["income3"].replace({2:1, 3:1,
                                                   4:2, 5:2, 6:2,
                                                   7:3, 8:3,
                                                   9:4, 10:4,
                                                   11:5,
                                                   77:np.nan, 99: np.nan})

# CHILDREN (Children in home indicator): Set any positive number of children to 1. 
                                       # Set No children (88) to None (0)
                                       # Set Refused (99) to missing  
brfss_df.loc[brfss_df['children'].between(1, 87), 'children'] = 1
brfss_df["children"] = brfss_df["children"].replace({88:0,
                                                     99:np.nan})

# _STATE (FIPS code): No Missingness

C:\Users\samue\AppData\Local\Temp\ipykernel_30200\3732276881.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  brfss_df['addepev3'] = brfss_df['addepev3'].replace({2:0,
C:\Users\samue\AppData\Local\Temp\ipykernel_30200\3732276881.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  brfss_df["firearm5"] = brfss_df["firearm5"].replace({2:0,
C:\Users\samue\AppData\Local\Temp\ipykernel_30200\3732276881.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using 

In [6]:
# Copy cleaned version of dataset
brfss_clean_df = brfss_df.copy()

# Examine distribution of each variable (including missings)
print(brfss_clean_df['addepev3'].value_counts(normalize = True, dropna = False))
print(brfss_clean_df["firearm5"].value_counts(normalize = True, dropna = False))
print(brfss_clean_df["loadulk2"].value_counts(normalize = True, dropna = False))
print(brfss_clean_df["_ment14d"].value_counts(normalize = True, dropna = False))
print(brfss_clean_df['sexvar'].value_counts(normalize = True, dropna = False))
print(brfss_clean_df['_age80'].describe())
print(brfss_clean_df["_racegr3"].value_counts(normalize = True, dropna = False))
print(brfss_clean_df["primins1"].value_counts(normalize = True, dropna = False))
print(brfss_clean_df["educa"].value_counts(normalize = True, dropna = False))
print(brfss_clean_df["income3"].value_counts(normalize = True, dropna = False))
print(brfss_clean_df["children"].value_counts(normalize = True, dropna = False))

addepev3
0.0    0.790604
1.0    0.203426
NaN    0.005970
Name: proportion, dtype: float64
firearm5
NaN    0.827528
0.0    0.102755
1.0    0.069717
Name: proportion, dtype: float64
loadulk2
NaN    0.976821
1.0    0.012679
0.0    0.010500
Name: proportion, dtype: float64
_ment14d
1.0    0.593151
2.0    0.254129
3.0    0.134009
NaN    0.018711
Name: proportion, dtype: float64
sexvar
2    0.529723
1    0.470277
Name: proportion, dtype: float64
count    433323.000000
mean         55.375461
std          17.909598
min          18.000000
25%          41.000000
50%          58.000000
75%          71.000000
max          80.000000
Name: _age80, dtype: float64
_racegr3
1.0    0.722480
5.0    0.099893
2.0    0.075969
3.0    0.056207
4.0    0.023366
NaN    0.022085
Name: proportion, dtype: float64
primins1
1.0    0.429603
2.0    0.383631
3.0    0.091278
4.0    0.052393
NaN    0.043095
Name: proportion, dtype: float64
educa
3.0    0.426626
1.0    0.304126
2.0    0.263882
NaN    0.005366
Name: proport

# ***Contingency Table (1) Prevalence of Depression among Gun Owners***

In [7]:
# Crosstab between depression and firearem ownership
depr_firearms_crosstab = pd.crosstab(brfss_clean_df['addepev3'], brfss_clean_df['firearm5'], margins = True)
print(depr_firearms_crosstab)

# Proportions
depr_firearms_prop = depr_firearms_crosstab = pd.crosstab(brfss_clean_df['addepev3'], brfss_clean_df['firearm5'], normalize = 'columns') * 100
print(depr_firearms_prop)

# P-value overall
chisq_pvalue = chi2_contingency(pd.crosstab(brfss_clean_df['addepev3'], brfss_clean_df['firearm5'])).pvalue
print(chisq_pvalue)

firearm5    0.0    1.0    All
addepev3                     
0.0       33848  23767  57615
1.0       10399   6296  16695
All       44247  30063  74310
firearm5        0.0        1.0
addepev3                      
0.0       76.497842  79.057313
1.0       23.502158  20.942687
2.487206276707508e-16


20.9% of gun owners have had depression compared with 23.5% of non-gun owners. The difference in depression between gun owners and non-owners is statistically significant at the 5% level (p < 0.001). 

# ***Logistic Regression (2a) Depression ~ Gun Ownership + covariates***

In [54]:
# Outcome: ADDEPEV3 (Depression)
# Main predictor: FIREARM5 (Gun ownership)
# Covariates: _AGE80, SEXVAR, _RACEGR3, EDUCA, INCOME3, PRIMINS1, 

# Logistic Regression
data_2a = brfss_clean_df[['addepev3', 'firearm5', '_age80', 'sexvar', '_racegr3', 'educa', 'income3', 'primins1']].dropna().copy() 
logreg_2a = 'addepev3 ~ C(firearm5) + _age80 + C(sexvar) + C(_racegr3) + C(educa) + C(income3) + C(primins1)'
model_2a = smf.logit(formula = logreg_2a, data = data_2a)
results_2a = model_2a.fit()

# Show results
print(results_2a.summary())

Optimization terminated successfully.
         Current function value: 0.502693
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:               addepev3   No. Observations:                59677
Model:                          Logit   Df Residuals:                    59660
Method:                           MLE   Df Model:                           16
Date:                Sat, 22 Nov 2025   Pseudo R-squ.:                 0.06869
Time:                        16:53:07   Log-Likelihood:                -29999.
converged:                       True   LL-Null:                       -32212.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              0.4074      0.051      7.947      0.000       0.307       0.508
C(fir

Owning a gun is associated with...

# ***Logistic Regression (2b) Mental Health Status ~ Gun Ownership + covariates***

In [ ]:
# Outcome: _MENT14D (Mental Health Status)
# Main predictor: FIREARM5 (Gun ownership)
# Covariates: _AGE80, SEXVAR, _RACEGR3, EDUCA, INCOME3, PRIMINS1, 


# ***Logistic Regression (3a) Unsafe Gun Storage Practices ~ Depression + covariates***

In [ ]:
# Outcome: LOADULK2 (Unsafe storage)
# Main predictor: ADDEPEV3 (0/1 Depression)
# Covariates: _age80, sexvar, _racegr3, educa, income3, primins1, children

# keep only the respondents who have guns 
gun_df = df[df["firearm5"] == 1].copy()
print(gun_df["firearm5"].value_counts(dropna=False))

vars_2a = ["loadulk2", "addepev3", "_age80", "sexvar", "_racegr3", "educa", "income3", "primins1", "children"] #all vars needed for model 2a

m2a = gun_df[vars_3a].dropna().copy() # drop rows with any missings in these variables

#Logistirc Regression
logreg_2a = "loadulk2 ~ addepev3 + C(_age80) + C(sexvar) + C(_racegr3) + C(educa) + C(income3) + C(primins1) + C(children)"

model_2a = smf.logit(formula=logreg_2a, data=m2a) #https://www.statsmodels.org/stable/generated/statsmodels.formula.api.logit.html
result_2a = model_2a.fit()

print(result_2a.summary())

# ***Logistic Regression (3b) Unsafe Gun Storage Practices ~ Poor Mental Health Status + covariates***